# Sentinel-3 land surface temperature over Berlin

This notebook demonstrates the main workflows provided by `sentinel-analysis`:

- query the official CDSE OData catalogue for Sentinel-3 SLSTR L2 LST products;
- download and inspect real SAFE products;
- read baseline-independent NetCDF layouts with `xarray`;
- apply the documented L2 LST exception flags;
- convert Kelvin to Celsius and aggregate observations by day;
- plot the observed surface-temperature evolution over Berlin;
- inspect the optional openEO and EODAG provider paths.

The example selects at most one low-cloud NTC product per day over a two-week period.
Sentinel-3 is an overpass sensor, so the resulting time series contains
irregular satellite observations rather than a continuous hourly record.
The local CDSE download flow requires `CDSE_USERNAME` and `CDSE_PASSWORD`
in `.env`; client credentials are used by the openEO workflow.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

# Make the notebook work from the repository root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sentinel_analysis import (
    AOI,
    ClientConfig,
    ProductQuery,
    QualityPolicy,
    Sentinel3LST,
    Sentinel3LSTClient,
    apply_quality_mask,
    daily_mean,
    inspect_l2_lst,
    quality_summary,
    read_l2_lst,
    statistics,
    to_celsius,
)
from sentinel_analysis.sensors.sentinel3.processing import clip_bbox
from sentinel_analysis.sensors.sentinel3.quality import exception_counts

OUTPUT_DIR = PROJECT_ROOT / 'output'
DOWNLOAD_DIR = OUTPUT_DIR / 'berlin_lst_products'
EXTRACT_DIR = OUTPUT_DIR / 'berlin_lst_extracted'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

BERLIN_AOI = AOI(west=13.20, south=52.40, east=13.55, north=52.58)
START_DATE = '2025-06-10'
END_DATE = '2025-06-24'
MAX_DAYS = 14
MAX_CATALOG_CLOUD_COVER = 30.0
# In the SLSTR cloud_in flags, bit 7 means gross_cloud.
GROSS_CLOUD_FLAG = 1 << 7

print(f'Project root: {PROJECT_ROOT}')
print(f'Berlin extent: {BERLIN_AOI.as_extent()}')
print(f'Analysis period: {START_DATE} to {END_DATE}')

## 1. Input validation and query construction

`AOI` validates the WGS84 bounding box and can export it as an extent, WKT,
or GeoJSON. `ProductQuery` normalizes dates for the CDSE overlap query; a
date-only end value includes the complete end day.

In [ ]:
query = ProductQuery(
    aoi=BERLIN_AOI,
    start=START_DATE,
    end=END_DATE,
    timeliness='NTC',
)

print('Extent:', BERLIN_AOI.as_extent())
print('WKT:', BERLIN_AOI.as_wkt())
print('GeoJSON:', BERLIN_AOI.as_geojson())
print('Normalized dates:', query.normalized_dates())

## 2. Optional openEO graph workflow

The original openEO client remains available for server-side processing.
This cell builds a graph only: it does not submit a job or download data.
The graph loads LST, converts it to Celsius, and creates daily temporal
aggregation intervals.

In [ ]:
try:
    remote_client = Sentinel3LSTClient.from_env(PROJECT_ROOT / '.env')
    remote_cube = remote_client.processed_cube(
        BERLIN_AOI, [START_DATE, END_DATE]
    )
    remote_daily = remote_client.daily_mean(
        remote_cube, START_DATE, END_DATE
    )
    print('openEO graph processes:')
    print([node['process_id'] for node in remote_daily.flat_graph().values()])
    display(remote_daily.flat_graph())
except Exception as exc:
    print(f'openEO graph skipped: {type(exc).__name__}: {exc}')

## 3. Search the real CDSE catalogue

The high-level `Sentinel3LST` facade searches the official `SL_2_LST___`
collection. We select the least-cloudy NTC product per acquisition day and
keep products with catalogue cloud cover at or below 30 percent. This is
an explicit acquisition filter, not a temperature-range assumption.

In [ ]:
local_client = Sentinel3LST.from_cdse()
products = local_client.search(
    BERLIN_AOI,
    START_DATE,
    END_DATE,
    timeliness='NTC',
    limit=100,
)

products_by_day = {}
for product in products:
    day = (product.start_datetime or product.name)[0:10]
    cloud_cover = product.metadata.get('attributes', {}).get('cloudCover')
    if cloud_cover is not None and float(cloud_cover) <= MAX_CATALOG_CLOUD_COVER:
        products_by_day.setdefault(day, []).append(product)

selected_products = [
    min(
        candidates,
        key=lambda product: float(product.metadata['attributes']['cloudCover']),
    )
    for day, candidates in sorted(products_by_day.items())
]
selected_products = selected_products[:MAX_DAYS]

print(f'Catalogue matches: {len(products)}')
print(f'Selected low-cloud products for {len(selected_products)} days:')
for product in selected_products:
    cloud_cover = product.metadata['attributes'].get('cloudCover')
    print(f'  {product.start_datetime}  cloud={cloud_cover:.1f}%  {product.name}')

## 4. Download, read, quality-mask, and aggregate

Each selected product is downloaded once and cached locally. The reader
discovers variables from the SAFE NetCDF files instead of relying on one
baseline-specific filename. The quality policy rejects documented L2 LST
exception flags, the product's gross_cloud flag, and non-finite LST values.
No arbitrary Celsius range is
applied. The geographic clip happens before statistics are computed.
Some days can still be missing because a clear product was not available.

In [ ]:
observations = []
downloaded_archives = []
extracted_products = []
latest_daily = None
latest_quality = None

for product in selected_products:
    archive = local_client.download(product, DOWNLOAD_DIR)
    downloaded_archives.append(archive)
    extract_dir = EXTRACT_DIR / product.product_id

    raw = read_l2_lst(archive, extract_dir=extract_dir)
    extracted_products.append(next(extract_dir.glob('*.SEN3')))
    berlin = clip_bbox(
        raw,
        BERLIN_AOI.west,
        BERLIN_AOI.south,
        BERLIN_AOI.east,
        BERLIN_AOI.north,
    )
    quality_policy = QualityPolicy(legacy_confidence_cloud_bit=GROSS_CLOUD_FLAG)
    quality = apply_quality_mask(berlin, quality_policy)
    celsius = to_celsius(quality)
    summary = quality_summary(celsius)

    if summary['pixels_valid'] == 0:
        print(f'Skipping {product.name}: no valid Berlin pixels')
        continue

    daily = daily_mean(celsius, assume_quality_masked=True)
    stats = statistics(daily)
    timestamp = np.datetime64(daily.time.values[0])
    observations.append({
        'time': timestamp,
        'mean_celsius': stats['mean'],
        'min_celsius': stats['min'],
        'max_celsius': stats['max'],
        'pixels_valid': summary['pixels_valid'],
        'product': product.name,
    })
    latest_daily = daily
    latest_quality = celsius
    print(
        f"{timestamp}: mean={stats['mean']:.2f} degC, "
        f"valid pixels={summary['pixels_valid']:,}"
    )

observations.sort(key=lambda row: row['time'])
assert observations, 'No valid observations were produced.'
print(f'Processed {len(observations)} real satellite observations.')

# For a larger workflow, the downloaded SAFE archives can also be combined:
# combined = Sentinel3LST.combine(downloaded_archives)

## 5. Berlin temperature evolution

The line connects the actual Sentinel-3 overpass observations. Gaps are
expected: this is not a gap-filled daily product. The shaded interval
shows the requested two-week analysis window.

In [ ]:
times = np.array([row['time'] for row in observations], dtype='datetime64[ns]')
means = np.array([row['mean_celsius'] for row in observations])
mins = np.array([row['min_celsius'] for row in observations])
maxs = np.array([row['max_celsius'] for row in observations])

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(times, mins, maxs, color='tab:orange', alpha=0.18, label='Berlin range')
ax.plot(times, means, 'o-', color='tab:red', linewidth=2, label='Mean LST')
ax.axvspan(
    np.datetime64(START_DATE),
    np.datetime64(END_DATE) + np.timedelta64(1, 'D'),
    color='tab:blue',
    alpha=0.05,
)
ax.set_title('Sentinel-3 SLSTR land surface temperature over Berlin')
ax.set_xlabel('Acquisition time (UTC)')
ax.set_ylabel('Land surface temperature (degC)')
ax.grid(True, alpha=0.25)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 6. Inspect one spatial snapshot

The same quality-masked data can be visualized spatially. This map uses
the last successfully processed observation in the selected period.

In [ ]:
if latest_daily is not None:
    image = latest_quality['lst'].isel(time=0)
    longitude = latest_quality['longitude'].isel(time=0)
    latitude = latest_quality['latitude'].isel(time=0)
    fig, ax = plt.subplots(figsize=(8, 6))
    valid = (
        np.isfinite(longitude.values)
        & np.isfinite(latitude.values)
        & np.isfinite(image.values)
    )
    mesh = ax.scatter(
        longitude.values[valid],
        latitude.values[valid],
        c=image.values[valid],
        s=35,
        marker='s',
        cmap='inferno',
    )
    fig.colorbar(mesh, ax=ax, label='LST (degC)')
    ax.set_title('Quality-masked Berlin snapshot')
    ax.set_xlabel('Longitude (deg)')
    ax.set_ylabel('Latitude (deg)')
    fig.tight_layout()
    plt.show()

## 7. Quality diagnostics and product inspection

`inspect_l2_lst` is useful before loading a product into an analysis.
`exception_counts` reports the official exception bits independently,
while `statistics` summarizes the quality-masked Celsius data.

In [ ]:
if extracted_products:
    print('SAFE inspection:')
    display(inspect_l2_lst(extracted_products[-1])['metadata'])

if latest_quality is not None:
    print('Latest Berlin quality summary:')
    print(quality_summary(latest_quality))
    print('Latest Berlin statistics (degC):')
    print(statistics(latest_quality))
    print('Exception counts:')
    print(exception_counts(latest_quality))

## 8. Optional EODAG adapter

EODAG is not required for the direct CDSE workflow. Install the optional
extra with `uv sync --extra cdse` if this provider abstraction is needed.

In [ ]:
USE_EODAG = False
if USE_EODAG:
    from sentinel_analysis.sensors.sentinel3.eodag_provider import EODAGProvider

    eodag = EODAGProvider(provider='cop_dataspace')
    eodag_results = eodag.search(
        BERLIN_AOI, START_DATE, END_DATE, timeliness='NTC'
    )
    print(f'EODAG results: {len(eodag_results)}')
else:
    print('EODAG example disabled; direct CDSE OData was used above.')

## Takeaways

This workflow keeps acquisition and local analysis separate: CDSE provides
the official products, the SAFE reader standardizes changing NetCDF layouts,
and `xarray` handles quality masking, unit conversion, clipping, aggregation,
and statistics. For a continuous daily temperature series, combine these
observations with a gap-aware analysis or a complementary daily product;
the notebook intentionally does not interpolate missing satellite overpasses.